# Homework 4

### Problem 5.2

I am using Python to help me calculate the stationary points of this function since it is not super fun to do by hand.

In [21]:
import sympy as sp
import numpy as np

In [22]:
x_1, x_2 = sp.symbols('x_1 x_2')

f_1 = -13 + x_1 + ((5-x_2)*x_2-2)*x_2
f_2 = -29 + x_1 + ((x_2+1)*x_2-14)*x_2
f = f_1**2 + f_2**2

f_x_1 = sp.diff(f, x_1)
f_x_2 = sp.diff(f, x_2)

stationary_points = sp.solve([f_x_1, f_x_2], (x_1, x_2))
print(stationary_points)

[(5, 4), (13 - 14*I, -1 - I), (13 + 14*I, -1 + I), (53/3 - 4*sqrt(22)/3, 2/3 - sqrt(22)/3), (4*sqrt(22)/3 + 53/3, 2/3 + sqrt(22)/3)]


Now we need to calculate the Hessian to classify the points.

In [19]:
stationary_points = [(5,4), (53/3 - 4*sp.sqrt(22)/3, 2/3 - sp.sqrt(22)/3), (4*sp.sqrt(22)/3 + 53/3, 2/3 + sp.sqrt(22)/3)] # Retyping to remove the imaginary values

hessian = sp.hessian(f, (x_1, x_2))

for point in stationary_points:
    H_point = hessian.subs({x_1: point[0], x_2: point[1]})
    det = H_point.det()
    f_xx = H_point[0, 0]
    x_1_val, x_2_val = point
    
    print(f"\nClassification for point {point}:")
    f_value = f.subs({x_1: x_1_val, x_2: x_2_val})
    print(f"Value of the function is:{f_value}")
    if det > 0:
        if f_xx > 0:
            print("-> Local Minimum")
        else:
            print("-> Local Maximum")
    elif det < 0:
        print("-> Saddle Point")
    else:
        print("-> Inconclusive (Higher-order test needed)")



Classification for point (5, 4):
Value of the function is:0
-> Local Minimum

Classification for point (17.6666666666667 - 4*sqrt(22)/3, 0.666666666666667 - sqrt(22)/3):
Value of the function is:(-11.3333333333333 - 4*sqrt(22)/3 + (-14 + (0.666666666666667 - sqrt(22)/3)*(1.66666666666667 - sqrt(22)/3))*(0.666666666666667 - sqrt(22)/3))**2 + (-4*sqrt(22)/3 + 4.66666666666667 + (0.666666666666667 - sqrt(22)/3)*((0.666666666666667 - sqrt(22)/3)*(sqrt(22)/3 + 4.33333333333333) - 2))**2
-> Local Minimum

Classification for point (4*sqrt(22)/3 + 17.6666666666667, 0.666666666666667 + sqrt(22)/3):
Value of the function is:(4.66666666666667 + 4*sqrt(22)/3 + (-2 + (0.666666666666667 + sqrt(22)/3)*(4.33333333333333 - sqrt(22)/3))*(0.666666666666667 + sqrt(22)/3))**2 + ((-14 + (0.666666666666667 + sqrt(22)/3)*(sqrt(22)/3 + 1.66666666666667))*(0.666666666666667 + sqrt(22)/3) - 11.3333333333333 + 4*sqrt(22)/3)**2
-> Saddle Point


Now we will use three different numerical methods and compare them. The starting points we will evaluate at are:
$$(-50,7)^T, (20,7)^T, (20,-18)^T, (5,-10)^T$$
The first method we will use is Gradient Method with Backtracking and parameters $(s,\alpha,\beta)=(1,0.5,0.5)$. I am copying and pasting code that I already had created in-class for this method.

In [34]:
# Rewrite the function to be compatible with code already written
def f(x):
    """
    Function to minimize: f(x, y) = x^2 + 2*y^2
    x is a numpy array [x_val, y_val]
    """
    return (-13 + x[0] + ((5-x[1])*x[1]-2)*x[1])**2 + (-29 + x[0] + ((x[1]+1)*x[1]-14)*x[1])**2

def nabla_f(x):
    """
    Gradient of the function: nabla_f(x, y) = [2x, 4y]
    x is a numpy array [x_val, y_val]
    """
    return np.array([2*(-13 + x[0] + ((5-x[1])*x[1]-2)*x[1])+2*(-29 + x[0] + ((x[1]+1)*x[1]-14)*x[1]), 2*(-13 + x[0] + ((5-x[1])*x[1]-2)*x[1])*(10*x[1]-3*x[1]**2-2) + 2*(-29 + x[0] + ((x[1]+1)*x[1]-14)*x[1])*(3*x[1]**2+2*x[1]-14)])

TOL = 10e-5

In [36]:
x0 = np.array([-50.0, 7.0])
#x0 = np.array([-20.0, 7.0])
#x0 = np.array([-20.0, -18.0])
#x0 = np.array([5.0, -10.0])

def backtracking_line_search_gd(x_start, tol, max_iter=10000, alpha_init=1.0, rho=0.5, c=0.5):
    x = np.copy(x_start)
    path = [x.copy()] # Store the initial point
    for i in range(max_iter):
        grad = nabla_f(x)
        grad_norm = np.linalg.norm(grad)
        if grad_norm < tol:
            break

        alpha = alpha_init
        grad_norm_sq = grad_norm**2
        # Armijo condition: f(x - alpha * grad) <= f(x) - c * alpha * ||grad||^2
        while f(x - alpha * grad) > f(x) - c * alpha * grad_norm_sq:
            alpha *= rho
            if alpha < 1e-12: # Safeguard against tiny alpha causing stagnation
                break # Break inner while loop

        if alpha < 1e-12: # If alpha became too small, consider it converged or stuck
            break # Break outer for loop

        x_new = x - alpha * grad
        x = x_new
        path.append(x.copy())
    return np.array(path), i + 1

# Using common backtracking parameters
alpha_init_bt = 1.0
rho_bt = 0.5
c_bt = 0.5
path_backtracking, iters_backtracking = backtracking_line_search_gd(
    x0, TOL, alpha_init=alpha_init_bt, rho=rho_bt, c=c_bt
)
x_backtracking = path_backtracking[-1]
print(f"Backtracking parameters: alpha_init={alpha_init_bt}, rho={rho_bt}, c={c_bt}")
print(f"Converged at x: {x_backtracking}")
print(f"Function value at x: {f(x_backtracking)}")
print(f"Iterations: {iters_backtracking}\n")

Backtracking parameters: alpha_init=1.0, rho=0.5, c=0.5
Converged at x: [4.99996587 4.00000058]
Function value at x: 1.689658077487543e-09
Iterations: 1966



The next method we will use is Hybrid Newton's Method with parameters $(\alpha,\beta)=(0.5,0.5)$.

In [ ]:
def hybrid_newton(x_start, alpha, beta, tol=TOL, max_iterations=1000):
    x = np.copy(x_start)
    path = [x.copy()]
    for i in range(max_iterations):
        